# aam-p0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/federicopeinado/aam-p0/blob/main/aam-p0.ipynb)

**Autor** Federico Peinado
**Entorno:** Python 3.x (Sin bibliotecas de terceros como Pandas, NumPy o Matplotlib)

## Contexto

Partimos de un conjunto de datos de telemetría sobre 150 partidas al juego de las Magic. Hay que analizar si se produce alguno de estos desequilibrios en el juego.
1. El mazo **Mono-Red Aggro** gana demasiadas partidas.
2. El jugador que sale primero (**On the Play**) tiene una ventaja injusta frente al que sale segundo (**On the Draw**).

El objetivo es procesar este conjunto de datos a mano utilizando **Python puro** (sin `pandas`, `numpy`, `scipy` o `matplotlib`, pero con `json`, `math`, etc.) para confirmar o desmentir estadísticamente estas afirmaciones y proponer una posible solución de diseño para reequilibrar el juego.

## A. Carga correcta del conjunto de datos desde fichero, según su formato y condiciones.

Necesitarás importar la biblioteca `json` y es buena idea dar un mensaje de confirmación de la cantidad de muestras cargadas, incluso mostrando las primeras para ver el aspecto de los datos. Interesante darse cuenta de que no está contenida la información completa de las partidas sino sólo la información relativa a algunos jugadores rastreados (el héroe, según el conjunto de datos)

*Si ejecutas este cuaderno interactivo en Google Colab, la siguiente celda opcional descarga automáticamente el conjunto de datos desde el repositorio:*

In [3]:
# [OPCIONAL PARA COLAB] Descarga del conjunto de datos directamente desde GitHub
!wget https://raw.githubusercontent.com/federicopeinado/aam-p0/main/match_logs.json

--2026-09-09 23:58:54--  https://raw.githubusercontent.com/federicopeinado/aam-p0/main/match_logs.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28131 (27K) [text/plain]
Saving to: ‘match_logs.json’

match_logs.json     100%[===================>]  27.47K  --.-KB/s    in 0s      

2026-09-09 23:58:54 (113 MB/s) - ‘match_logs.json’ saved [28131/28131]



In [4]:
import json

# Carga del fichero de telemetría
file_path = "match_logs.json"

# Siempre fundamental la codificación para no tener problemas en distintos sistemas operativos
# Lo que vamos a tener en match_logs, aunque no se indique el tipo, es una lista
with open(file_path, "r", encoding="utf-8") as f:
    match_logs = json.load(f)

# Confirmación de la carga
total_muestras = len(match_logs)
print(f"==================================================")
print(f"   CARGA DE DATOS COMPLETADA")
print(f"==================================================")
print(f"Se han cargado correctamente {total_muestras} muestras de telemetría.\n")

print("Muestra de las 2 primeras partidas del dataset:")
# [:2] significa quedarse con las dos primeras muestras, la 0 y la 1
# enumerate devuelve tuplas índice-elemento, empezando a contar desde 1 (en vez de desde 0 que es lo habitual)
for i, log in enumerate(match_logs[:2], 1):
    print(f"\n--- Registro #{i} ---")
    # Esto recorre todos los pares atributo-valor de cada muestra
    for key, value in log.items():
        print(f"  {key:<18}: {value}") # :<18 es un tema de formateo de cadenas, significa alinear a la izquierda ocupando siempre 18 caracteres

   CARGA DE DATOS COMPLETADA
Se han cargado correctamente 150 muestras de telemetría.

Muestra de las 2 primeras partidas del dataset:

--- Registro #1 ---
  match_id          : 17L_MTGA_78001
  deck_archetype    : Mono-Red Aggro
  on_the_play       : True
  game_length_turns : 4
  is_winner         : True
  hero_life_at_end  : 4

--- Registro #2 ---
  match_id          : 17L_MTGA_78002
  deck_archetype    : Domain Overlord
  on_the_play       : True
  game_length_turns : 6
  is_winner         : True
  hero_life_at_end  : 7


## B. Limpieza básica del conjunto de datos, eliminando los que no se necesitan.

Por problemas de conexión o abandono, hay partidas que pueden terminar en el primer turno de manera abrupta. Estas muestras pueden distorsionar los análisis y conviene eliminarlas.
Recuerda dar siempre realimentación de las operaciones que hagas sobre los datos, para saber los que quedan en el conjunto, por ejemplo.

In [5]:
# Criterio: Filtrar partidas con duración < 2 turnos (serán concesiones inmediatas del jugador o desconexiones)
# En una sóla línea de código, se crea una nueva lista (por compresión de listas) que contiene únicamente los registros que cumplen la condición
clean_logs = [log for log in match_logs if log.get("game_length_turns", 0) >= 2]

muestras_descartadas = total_muestras - len(clean_logs)
porcentaje_descartado = (muestras_descartadas / total_muestras) * 100

print(f"==================================================")
print(f"   PROCESO DE LIMPIEZA DE DATOS                   ")
print(f"==================================================")
print(f"Muestras iniciales         : {total_muestras}")
print(f"Muestras descartadas (< 2t): {muestras_descartadas} ({porcentaje_descartado:.2f}%)")
print(f"Muestras válidas finales   : {len(clean_logs)}")

   PROCESO DE LIMPIEZA DE DATOS                   
Muestras iniciales         : 150
Muestras descartadas (< 2t): 3 (2.00%)
Muestras válidas finales   : 147


## C. Exploración del conjunto de datos, visualizando las victorias en función del tipo de mazo, o si se empieza jugando o no.

Hay que agrupar los datos de alguna manera, posiblemente en diccionarios, para poder analizarlos. Realmente es un simple conteo en el que nos interesan especialmente aquellas partidas jugadas y ganadas según el arquetipo que buscamos o según sea cierto el atributo `on_the_play`.  

In [6]:
# 1. Agrupación por arquetipo de mazo
by_deck = {}
# 2. Agrupación por condición de salida (On the Play vs On the Draw)
# Es un diccionario con diccionarios anidados a su vez
by_position = {True: {"total": 0, "wins": 0}, False: {"total": 0, "wins": 0}}

for log in clean_logs:
    deck = log["deck_archetype"]
    won = log["is_winner"]
    on_play = log["on_the_play"]

    # Conteo por mazo
    if deck not in by_deck:
        # Cada vez que aparce un mazo nuevo le monto esta estructura de diccionario con los contadores inicializados a 0
        by_deck[deck] = {"total": 0, "wins": 0}
    by_deck[deck]["total"] += 1
    if won:
        by_deck[deck]["wins"] += 1

    # Conteo por posición de salida
    by_position[on_play]["total"] += 1
    if won:
        by_position[on_play]["wins"] += 1

print("=========================================================")
print("   EXPLORACIÓN VISUAL (con GRÁFICOS DE BARRAS UNICODE)   ")
print("=========================================================\n")

print("--- 1. Distribución de victorias por mazo ---")
for deck, data in by_deck.items(): # items() devuelve una lista de tuplas (clave, valor) y el bucle puede iterar sobre ellas
    # En Python pueden escribirse condicionales en una sola línea, en plan A if condición else B
    wr = (data["wins"] / data["total"]) * 100 if data["total"] > 0 else 0
    bar = "█" * int(wr / 2) # Cada bloque va a representar un 2%, el operador * en este caso repite la cadena tantas veces como el número por el que 'multipliquemos'
    print(f"{deck:<18} | {bar:<30} {wr:.1f}% ({data['wins']}/{data['total']})") # :.1f redonde al vuelo a un sólo decimal

print("\n--- 2. Victorias según posición de salida ---")
labels = {True: "On the Play (1º)", False: "On the Draw (2º)"}
for pos, data in by_position.items():
    wr = (data["wins"] / data["total"]) * 100 if data["total"] > 0 else 0
    bar = "█" * int(wr / 2)
    print(f"{labels[pos]:<18} | {bar:<30} {wr:.1f}% ({data['wins']}/{data['total']})")

   EXPLORACIÓN VISUAL (con GRÁFICOS DE BARRAS UNICODE)   

--- 1. Distribución de victorias por mazo ---
Mono-Red Aggro     | ███████████████████████████████ 63.0% (17/27)
Domain Overlord    | █████████████████████████      50.0% (13/26)
Azorius Control    | ██████████████████████████     53.8% (14/26)
Dimir Midrange     | █████████████████████          43.8% (14/32)
Golgari Midrange   | ██████████████████████         44.4% (16/36)

--- 2. Victorias según posición de salida ---
On the Play (1º)   | ██████████████████████████     53.2% (42/79)
On the Draw (2º)   | ███████████████████████        47.1% (32/68)


## D. Análisis estadístico descriptivo del conjunto de datos en limpio, tratando de demostrar también si los datos dan la razón a los miembros de la comunidad que se han quedado de excesivas victorias de Mono Red Aggro y o del jugador que empieza primero.

Se mostrarán las típicas medidas estadísticas de media, mediana, etc. de duración de los turnos según mazo y otras cuestiones interesantes (para lo cual seguramente haga falta ordenar los datos). El análisis más importante aquí se hará con funciones auxiliares que calculen la tasa de victorias según el caso que nos interesa, para validar o desmentir las quejas de la comunidad.

Para la visualización con Python puro se pueden crear gráfico de barras sencillos usando el 'truco' de representarlos mediante caracteres Unicode.

In [7]:
# Funciones auxiliares para estadísticos descriptivos
def calcular_media(valores):
    return sum(valores) / len(valores) if valores else 0.0

def calcular_mediana(valores):
    if not valores:
        return 0.0
    # Para la mediana necesitamos ordenar los valores y luego tomar el valor del medio (o la media de los dos del medio si hay un número par de elementos)
    sorted_v = sorted(valores)
    n = len(sorted_v)
    mid = n // 2
    if n % 2 == 0:
        return (sorted_v[mid - 1] + sorted_v[mid]) / 2.0
    return float(sorted_v[mid]) # Es más consistente devolver siempre un float en esta función

def calcular_winrate(ganadas, totales):
    return (ganadas / totales) * 100 if totales > 0 else 0.0

# Extracción de duraciones por mazo
turns_by_deck = {}
for log in clean_logs:
    deck = log["deck_archetype"]
    if deck not in turns_by_deck:
        turns_by_deck[deck] = []
    turns_by_deck[deck].append(log["game_length_turns"])

print("============================================================")
print("   ESTADÍSTICA DESCRIPTIVA Y VALIDACIÓN DE HIPÓTESIS")
print("==================================================\n")

print(f"{'Arquetipo Mazo':<18} | {'Winrate (%)':<11} | {'Media Turnos':<12} | {'Mediana Turnos':<14}")
print("-" * 65) # Pintamos 65 rallitas...

for deck, data in by_deck.items():
    wr = calcular_winrate(data["wins"], data["total"])
    turns = turns_by_deck[deck]
    media_t = calcular_media(turns)
    mediana_t = calcular_mediana(turns)
    print(f"{deck:<18} | {wr:>10.2f}% | {media_t:>12.2f} | {mediana_t:>14.1f}")

# Métricas globales On the Play vs On the Draw
wr_play = calcular_winrate(by_position[True]["wins"], by_position[True]["total"])
wr_draw = calcular_winrate(by_position[False]["wins"], by_position[False]["total"])

print("\n--------------------------------------------------")
print("EFECTO DE LA POSICIÓN INICIAL:")
print(f"  • Winrate On the Play (Empezar 1º) : {wr_play:.2f}%")
print(f"  • Winrate On the Draw (Empezar 2º) : {wr_draw:.2f}%")
print(f"  • Sesgo / Ventaja del 1er jugador : +{wr_play - wr_draw:.2f}%")
print("--------------------------------------------------")


   ESTADÍSTICA DESCRIPTIVA Y VALIDACIÓN DE HIPÓTESIS

Arquetipo Mazo     | Winrate (%) | Media Turnos | Mediana Turnos
-----------------------------------------------------------------
Mono-Red Aggro     |      62.96% |         5.11 |            5.0
Domain Overlord    |      50.00% |         8.69 |            9.0
Azorius Control    |      53.85% |        15.27 |           15.5
Dimir Midrange     |      43.75% |         9.56 |           10.0
Golgari Midrange   |      44.44% |         8.92 |            8.5

--------------------------------------------------
EFECTO DE LA POSICIÓN INICIAL:
  • Winrate On the Play (Empezar 1º) : 53.16%
  • Winrate On the Draw (Empezar 2º) : 47.06%
  • Sesgo / Ventaja del 1er jugador : +6.11%
--------------------------------------------------


> **Discusión sobre los resultados**  

Tras procesar los datos limpios de telemetría, se confirma que la primera queja de la comunidad está plenamente justificada por los datos, aunque en la segunda es cierto que hay ventaja aunque no es significativa:  

Mazo Mono-Red Aggro: Presenta un Winrate superior al resto de arquetipos (superando holgadamente el 60%), con una duración de partida notablemente más corta (media de 5 turnos). Esto demuestra que es un mazo excesivamente dominante y rápido que rompe el equilibrio esperado entre arquetipos.

Ventaja del primer jugador (On the Play): El Winrate del jugador que inicia la partida es de un 53%, mientras que el del segundo jugador es del 47%. La diferencia no es tan grande cuando se nos dice en el enunciado que sería tolerable hasta un umbral máximo de tolerancia del 55% de tasa de victorias, de modo que la situación no es problemática.

## E. Propuesta final para mejorar los resultados en caso de que se haya probado que existen los desequilibrios mencionados.

Esta es la parte más libre de todo el enunciado. En caso de que se hayan encontrado desequilibrios, algo que se podría hacer es clasificar los mazos según su nivel de 'desequilibrio' y en base a eso proponer medidas para corregir los problemas encontrados (nerfear o bufear dichos mazos), así como intentar averiguar si el sesgo de ganar jugar primero tiene algo que ver con que las partidas sean más cortas o no.  

In [8]:
def clasificar_balanceo(winrate):
    """Clasifica un arquetipo según su nivel de desequilibrio."""
    if winrate > 58.0:
        return "Demasiado poderoso (require NERF)"
    elif winrate < 42.0:
        return "Demasiado débil (requiere BUFF)"
    else:
        return "Equilibrado"

print("==================================================")
print("   INFORME AUTOMATIZADO DE EQUILIBRADO            ")
print("==================================================\n")

print("--- Diagnóstico por Arquetipos ---")
for deck, data in by_deck.items():
    wr = calcular_winrate(data["wins"], data["total"])
    estado = clasificar_balanceo(wr)
    print(f"• {deck:<18} (Winrate: {wr:.1f}%) -> Estado: {estado}")

# Análisis cruzado: Duración vs Posición para entender mejor la ventaja de salir primero, aunque es pequeña
turns_play = [log["game_length_turns"] for log in clean_logs if log["on_the_play"]]
turns_draw = [log["game_length_turns"] for log in clean_logs if not log["on_the_play"]]

print("\n--- Relación entre duración y ventaja del Primer Jugador ---")
print(f"• Duración media On the Play : {calcular_media(turns_play):.2f} turnos")
print(f"• Duración media On the Draw : {calcular_media(turns_draw):.2f} turnos")

   INFORME AUTOMATIZADO DE EQUILIBRADO            

--- Diagnóstico por Arquetipos ---
• Mono-Red Aggro     (Winrate: 63.0%) -> Estado: Demasiado poderoso (require NERF)
• Domain Overlord    (Winrate: 50.0%) -> Estado: Equilibrado
• Azorius Control    (Winrate: 53.8%) -> Estado: Equilibrado
• Dimir Midrange     (Winrate: 43.8%) -> Estado: Equilibrado
• Golgari Midrange   (Winrate: 44.4%) -> Estado: Equilibrado

--- Relación entre duración y ventaja del Primer Jugador ---
• Duración media On the Play : 9.05 turnos
• Duración media On the Draw : 9.90 turnos


> **Propuesta final**  

Para solucionar el desequilibrio de Mono-Red Aggro, se propone reducir la potencia de sus cartas de inicio o reducir el daño directo que puede hacer con bajo coste (aumentando el coste de esas cartas o reduciendo la fuerza de sus criaturas en los primeros turnos). Esto debería retrasar su curva de ataque 1-2 turnos, dando margen a los mazos de Control y Midrange para estabilizar la mesa... cosa que por supuesto, tendremos que comprobar en un análisis posterior.

Con respecto al sesgo estructural de salir On the Play, no se recomienda realizar ningún cambio por el momento y tampoco se aprecia que haya mucha diferencia entre la duración de las partidas sea que el jugador empiece o que juegue en segundo lugar (ni siquiera llega a un turno de diferencia).